# Build Train/Test Datasets with Target and Control Patients

This notebook orchestrates building train/test datasets for final model training, ensuring **both target and control patients** are included in all splits.

## Overview

This notebook runs the complete dataset preparation pipeline:

1. **Build Final Feature Table** - Merges all feature engineering outputs (FP-Growth, BupaR, DTW, PGx, Predictive Time)
2. **Remove Target Leakage** - Removes post-event features, time-to-target features, and DTW features
3. **Prepare Train/Test Splits** - Creates temporal validation splits (2016-2018 train, 2019 test) with both target and control

## Key Features

✅ **Both Target and Control Patients** - All feature engineering includes both patient types  
✅ **Temporal Validation** - Train on 2016-2018, test on 2019 (no data leakage)  
✅ **Target Leakage Removal** - Automatic removal of post-event and time-to-target features  
✅ **Idempotent Workflow** - Skips steps if outputs already exist  
✅ **Verification** - Validates that both target and control patients are in train/test splits  

## Output Files

- **Final Features**: `6_final_model/outputs/{cohort}/{age_band}/{cohort}_{age_band}_train_final_features_no_leakage.csv`
- **Train Dataset**: `6_final_model/outputs/{cohort}/{age_band}/inputs/model_train/final_features.parquet`
- **Test Dataset**: `6_final_model/outputs/{cohort}/{age_band}/inputs/model_test/final_features.parquet`
- **S3 Location**: `s3://pgxdatalake/gold/final_model/{cohort}/{age_band}/inputs/`


## Environment Setup


In [ ]:
import os
import sys
from pathlib import Path

# -------------------------------------------------------------
# Resolve Python binary for BOTH EC2 and local Windows
# -------------------------------------------------------------
def resolve_python_bin() -> Path:
    """
    Priority:
    1. COHORT_RUNNER_PYTHON env var (explicit override)
    2. Current kernel Python (sys.executable)
    """
    env_bin = os.environ.get("COHORT_RUNNER_PYTHON")
    if env_bin:
        return Path(env_bin)
    return Path(sys.executable)


PYTHON_BIN = resolve_python_bin()
print(f"[INFO] Using Python binary: {PYTHON_BIN}")

# -------------------------------------------------------------
# Resolve project_root robustly for BOTH notebook + script mode
# -------------------------------------------------------------
def resolve_project_root() -> Path:
    # Case 1: running as a script -> __file__ exists
    if "__file__" in globals():
        return Path(__file__).resolve().parents[1]

    # Case 2: running in Jupyter/Notebook -> no __file__
    notebook_path = Path(os.getcwd()).resolve()

    # If running in .../pgx-analysis/6_final_model, go up 1 level
    if notebook_path.name == "6_final_model":
        return notebook_path.parent

    # If running deeper inside scripts, go up until pgx-analysis appears
    for parent in notebook_path.parents:
        if parent.name == "pgx-analysis":
            return parent

    # Last fallback: use current working directory
    return notebook_path


PROJECT_ROOT = resolve_project_root()
print(f"[INFO] Project root: {PROJECT_ROOT}")

# Add to sys.path if needed
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))


## Using the Orchestration Script

For a complete workflow (build features + remove leakage + create splits), use the orchestration script:

```python
# Example: Run complete workflow for a cohort/age band
import subprocess

subprocess.run(
    [str(PYTHON_BIN), "6_final_model/run_final_model.py", 
     "--cohort", "opioid_ed", 
     "--age_band", "0-12",
     "--skip-training",  # Skip model training, only build datasets
     "--skip-visualizations"],  # Skip visualizations
    cwd=PROJECT_ROOT,
)
```

**Benefits:**
- Runs all dataset preparation steps in one command
- Can be called from notebooks or command line
- Includes verification automatically

**Alternative:** Use individual cells below for more control over specific steps.


## Per-Cohort Runner Cells

Each cell below runs the complete dataset preparation pipeline for a **single cohort/age-band combination**. This makes it easy to:

- Debug failures for a specific cohort/age-band
- Modify scripts and immediately re-run just that cohort
- Verify outputs for individual cohorts

All scripts are idempotent (skip completed steps if outputs already exist).


### Cohort 1 – Age 0–12


In [ ]:
# Cohort 1, Age 0-12

import subprocess

# Step 1: Build final feature table
print("=" * 80)
print("[STEP 1/3] Building final feature table...")
print("=" * 80)
result1 = subprocess.run(
    [str(PYTHON_BIN), "6_final_model/build_final_cohort_model_features.py",
     "--cohort", "opioid_ed", "--age_band", "0-12"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)
if result1.returncode == 0:
    print("✓ Feature table built successfully")
    if result1.stdout:
        print(result1.stdout)
else:
    print(f"✗ Feature table build failed: {result1.stderr}")
    raise RuntimeError("Feature table build failed")

# Step 2: Remove target leakage
print("\n" + "=" * 80)
print("[STEP 2/3] Removing target leakage...")
print("=" * 80)
result2 = subprocess.run(
    [str(PYTHON_BIN), "6_final_model/remove_target_leakage.py",
     "--cohort", "opioid_ed", "--age_band", "0-12"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)
if result2.returncode == 0:
    print("✓ Target leakage removed successfully")
    if result2.stdout:
        print(result2.stdout)
else:
    print(f"✗ Leakage removal failed: {result2.stderr}")
    raise RuntimeError("Leakage removal failed")

# Step 3: Prepare train/test splits
print("\n" + "=" * 80)
print("[STEP 3/3] Preparing train/test splits...")
print("=" * 80)
result3 = subprocess.run(
    [str(PYTHON_BIN), "6_final_model/prepare_train_test_s3.py",
     "--cohort", "opioid_ed", "--age_band", "0-12"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)
if result3.returncode == 0:
    print("✓ Train/test splits prepared successfully")
    if result3.stdout:
        print(result3.stdout)
else:
    print(f"✗ Train/test split failed: {result3.stderr}")
    raise RuntimeError("Train/test split failed")

print("\n" + "=" * 80)
print("✓ Dataset preparation complete for opioid_ed / 0-12")
print("=" * 80)


### Verify Train/Test Splits Include Both Target and Control

Use this cell to verify that both target and control patients are included in the train/test splits.


In [ ]:
# Verify train/test splits include both target and control patients

import pandas as pd

cohort_name = "opioid_ed"
age_band = "0-12"
age_band_fname = age_band.replace("-", "_")

# Load train and test datasets
train_path = PROJECT_ROOT / "6_final_model" / "outputs" / cohort_name / age_band_fname / "inputs" / "model_train" / "final_features.parquet"
test_path = PROJECT_ROOT / "6_final_model" / "outputs" / cohort_name / age_band_fname / "inputs" / "model_test" / "final_features.parquet"

if train_path.exists():
    train_df = pd.read_parquet(train_path)
    print("=" * 80)
    print("TRAIN DATASET VERIFICATION")
    print("=" * 80)
    print(f"Total patients: {len(train_df)}")
    if 'target' in train_df.columns:
        target_counts = train_df['target'].value_counts()
        print(f"\nTarget distribution:")
        print(f"  Target (1): {target_counts.get(1, 0)} patients")
        print(f"  Control (0): {target_counts.get(0, 0)} patients")
        print(f"  Ratio: {target_counts.get(0, 0) / max(target_counts.get(1, 1), 1):.2f}:1 (control:target)")
        
        if target_counts.get(0, 0) == 0:
            print("\n⚠ WARNING: No control patients in training set!")
        elif target_counts.get(1, 0) == 0:
            print("\n⚠ WARNING: No target patients in training set!")
        else:
            print("\n✓ Both target and control patients present in training set")
    else:
        print("⚠ WARNING: 'target' column not found in training set")
else:
    print(f"⚠ Training dataset not found: {train_path}")

if test_path.exists():
    test_df = pd.read_parquet(test_path)
    print("\n" + "=" * 80)
    print("TEST DATASET VERIFICATION")
    print("=" * 80)
    print(f"Total patients: {len(test_df)}")
    if 'target' in test_df.columns:
        target_counts = test_df['target'].value_counts()
        print(f"\nTarget distribution:")
        print(f"  Target (1): {target_counts.get(1, 0)} patients")
        print(f"  Control (0): {target_counts.get(0, 0)} patients")
        print(f"  Ratio: {target_counts.get(0, 0) / max(target_counts.get(1, 1), 1):.2f}:1 (control:target)")
        
        if target_counts.get(0, 0) == 0:
            print("\n⚠ WARNING: No control patients in test set!")
        elif target_counts.get(1, 0) == 0:
            print("\n⚠ WARNING: No target patients in test set!")
        else:
            print("\n✓ Both target and control patients present in test set")
    else:
        print("⚠ WARNING: 'target' column not found in test set")
else:
    print(f"⚠ Test dataset not found: {test_path}")

print("\n" + "=" * 80)
print("VERIFICATION COMPLETE")
print("=" * 80)


## Run All Cohorts Sequentially

Use this section to run dataset preparation for all cohorts/age bands sequentially.


In [ ]:
# Run dataset preparation for all cohorts sequentially

import subprocess
from pathlib import Path

# Configuration: which cohorts to run
COHORTS = [
    ("opioid_ed", "0-12"),
    ("opioid_ed", "13-24"),
    ("opioid_ed", "25-44"),
    ("opioid_ed", "45-54"),
    ("opioid_ed", "55-64"),
    ("non_opioid_ed", "65-74"),
    ("non_opioid_ed", "75-84"),
    ("non_opioid_ed", "85-94"),
]

FAIL_FAST = True  # Stop on first failure; set to False to continue on errors

for cohort_name, age_band in COHORTS:
    print("\n" + "=" * 80)
    print(f"Processing: {cohort_name} / {age_band}")
    print("=" * 80)
    
    # Step 1: Build final feature table
    print(f"\n[STEP 1/3] Building final feature table...")
    result1 = subprocess.run(
        [str(PYTHON_BIN), "6_final_model/build_final_cohort_model_features.py",
         "--cohort", cohort_name, "--age_band", age_band],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
    )
    if result1.returncode != 0:
        msg = f"Feature table build failed for {cohort_name} / {age_band}"
        print(f"✗ {msg}")
        if FAIL_FAST:
            raise RuntimeError(msg)
        continue
    
    # Step 2: Remove target leakage
    print(f"\n[STEP 2/3] Removing target leakage...")
    result2 = subprocess.run(
        [str(PYTHON_BIN), "6_final_model/remove_target_leakage.py",
         "--cohort", cohort_name, "--age_band", age_band],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
    )
    if result2.returncode != 0:
        msg = f"Leakage removal failed for {cohort_name} / {age_band}"
        print(f"✗ {msg}")
        if FAIL_FAST:
            raise RuntimeError(msg)
        continue
    
    # Step 3: Prepare train/test splits
    print(f"\n[STEP 3/3] Preparing train/test splits...")
    result3 = subprocess.run(
        [str(PYTHON_BIN), "6_final_model/prepare_train_test_s3.py",
         "--cohort", cohort_name, "--age_band", age_band],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
    )
    if result3.returncode != 0:
        msg = f"Train/test split failed for {cohort_name} / {age_band}"
        print(f"✗ {msg}")
        if FAIL_FAST:
            raise RuntimeError(msg)
        continue
    
    print(f"\n✓ Completed: {cohort_name} / {age_band}")

print("\n" + "=" * 80)
print("All cohorts processed (or skipped if already complete)")
print("=" * 80)


## Verify All Cohorts

Use this cell to verify that all cohorts have train/test splits with both target and control patients.


In [ ]:
# Verify all cohorts have train/test splits with both target and control

import pandas as pd

COHORTS = [
    ("opioid_ed", "0-12"),
    ("opioid_ed", "13-24"),
    ("opioid_ed", "25-44"),
    ("opioid_ed", "45-54"),
    ("opioid_ed", "55-64"),
    ("non_opioid_ed", "65-74"),
    ("non_opioid_ed", "75-84"),
    ("non_opioid_ed", "85-94"),
]

print("=" * 80)
print("VERIFICATION SUMMARY - All Cohorts")
print("=" * 80)

all_valid = True

for cohort_name, age_band in COHORTS:
    age_band_fname = age_band.replace("-", "_")
    train_path = PROJECT_ROOT / "6_final_model" / "outputs" / cohort_name / age_band_fname / "inputs" / "model_train" / "final_features.parquet"
    test_path = PROJECT_ROOT / "6_final_model" / "outputs" / cohort_name / age_band_fname / "inputs" / "model_test" / "final_features.parquet"
    
    print(f"\n{cohort_name} / {age_band}:")
    
    # Check train
    if train_path.exists():
        train_df = pd.read_parquet(train_path)
        if 'target' in train_df.columns:
            target_counts = train_df['target'].value_counts()
            n_target = target_counts.get(1, 0)
            n_control = target_counts.get(0, 0)
            print(f"  Train: {len(train_df)} patients (Target: {n_target}, Control: {n_control})")
            if n_target == 0 or n_control == 0:
                print(f"    ⚠ WARNING: Missing {'target' if n_target == 0 else 'control'} patients!")
                all_valid = False
        else:
            print(f"  Train: {len(train_df)} patients (⚠ 'target' column missing)")
            all_valid = False
    else:
        print(f"  Train: ⚠ Dataset not found")
        all_valid = False
    
    # Check test
    if test_path.exists():
        test_df = pd.read_parquet(test_path)
        if 'target' in test_df.columns:
            target_counts = test_df['target'].value_counts()
            n_target = target_counts.get(1, 0)
            n_control = target_counts.get(0, 0)
            print(f"  Test:  {len(test_df)} patients (Target: {n_target}, Control: {n_control})")
            if n_target == 0 or n_control == 0:
                print(f"    ⚠ WARNING: Missing {'target' if n_target == 0 else 'control'} patients!")
                all_valid = False
        else:
            print(f"  Test:  {len(test_df)} patients (⚠ 'target' column missing)")
            all_valid = False
    else:
        print(f"  Test:  ⚠ Dataset not found")
        all_valid = False

print("\n" + "=" * 80)
if all_valid:
    print("✓ All cohorts verified - both target and control patients present in all splits")
else:
    print("⚠ Some cohorts have issues - check warnings above")
print("=" * 80)
